In [ ]:
!pip install -q transformers[torch] datasets sentencepiece

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# Configurazione Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Carica il nuovo dataset generato dallo script consensus_v3
df = pd.read_csv("/kaggle/input/joke-sample/dataset_LPR_new_PPO_v2.csv")

# Se la colonna del testo si chiama 'jokeText' (come nei tuoi file precedenti)
# la rinominiamo in 'text' per comodità di HuggingFace
text_col = "jokeText" if "jokeText" in df.columns else "text"
df = df[[text_col, 'LPR_new']].rename(columns={text_col: "text", "LPR_new": "label"})

# Convertiamo in formato Dataset di HuggingFace
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1) # 10% per validazione

In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Carichiamo il modello per REGRESSIONE
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=1, 
    problem_type="regression"
).to(device)

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta_v2_reward_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate= 2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    report_to="none" # o "wandb" se lo usi
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

# Salvataggio Finale
model.save_pretrained("deberta-v2-humor-judge")
tokenizer.save_pretrained("deberta-v2-humor-judge")
print("Reward Model DeBERTa addestrato e salvato!")

In [ ]:
from transformers import pipeline

# Carica il modello appena addestrato
judge = pipeline("text-classification", model="deberta-v2-humor-judge", tokenizer="deberta-v2-humor-judge")

# Prova con due battute: una che sai essere buona e una pessima
test_jokes = [
    "Why don't scientists trust atoms? Because they make up everything!", 
    "The cat sat on the mat and it was not funny at all."
]

for joke in test_jokes:
    result = judge(joke)
    print(f"Battuta: {joke}")
    print(f"Voto Reward Model: {result[0]['score']}\n")

In [ ]:
import shutil
import os

# Nome della cartella che abbiamo creato
folder_to_zip = '/kaggle/working/deberta-v2-humor-judge'

# Nome del file zip che vogliamo creare
output_zip = 'deberta-v2-humor-judge-v2'

# Creazione dello zip
if os.path.exists(folder_to_zip):
    shutil.make_archive(output_zip, 'zip', folder_to_zip)
    print(f"Archivio {output_zip}.zip creato con successo!")
else:
    print(f"Errore: la cartella {folder_to_zip} non esiste.")